In [ ]:
!pip install -U langchain langchain-community langchain-groq
!pip install langchain-huggingface
!pip install faiss-cpu sentence-transformers
!pip install pypdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling l

In [ ]:
import os
import gradio as gr

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain_core.tools import tool

/tmp/ipykernel_720/3200876519.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
uploaded = files.upload()

Saving Angular.pdf to Angular.pdf


In [ ]:
pdf_file = list(uploaded.keys())[0]

print("Uploaded file:", pdf_file)

Uploaded file: Angular.pdf


In [ ]:
os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxxx"

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq connected successfully")

Groq connected successfully


In [ ]:
response = llm.invoke(
    "What skills are required for a Data Scientist?"
)

print(response.content)

## Data Scientist – Core Skill Set (2024)

Below is a concise, yet comprehensive, checklist of the **technical, analytical, and soft‑skill** capabilities that most hiring managers and senior data teams expect from a modern Data Scientist.  Think of it as a “skill matrix” you can use to audit yourself, plan a learning path, or build a hiring rubric.

| Category | Skill | Why It Matters | Typical Tools / Libraries | Proficiency Level* |
|----------|-------|----------------|---------------------------|--------------------|
| **Programming** | **Python** | De‑facto language for data wrangling, modeling, and production pipelines. | pandas, NumPy, scikit‑learn, TensorFlow/PyTorch, Jupyter, Dask, FastAPI | ✅ 3‑5 years (or equivalent projects) |
| | **R** | Strong for statistical analysis, reporting, and certain niche domains (bio‑informatics, epidemiology). | tidyverse, data.table, caret, Shiny, mlr3 | ✅ Comfortable with core packages |
| | **SQL** | Extracting, transforming, and aggregating 

In [ ]:
loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 15


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 30


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created")

FAISS vector database created


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created")

Retriever created


In [ ]:
query = "What skills are required for a Data Scientist?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print("Result", i + 1)
    print(doc.page_content)
    print("-" * 50)

Result 1
that
 
provides
 
MVC
 
architecture,
 
SPA
 
support,
 
routing,
 
services,
 
dependency
 
injection,
 
REST
 
support,
 
templates,
 
and
 
testing
 
facilities.
 
These
 
features
 
simplify
 
development
 
and
 
help
 
developers
 
build
 
responsive
 
web
 
and
 
mobile
 
applications.
--------------------------------------------------
Result 2
3.
 
The
 
view
 
displays
 
the
 
model
 
data.
 
4.
 
The
 
user
 
interacts
 
with
 
the
 
view.
 
5.
 
The
 
controller
 
processes
 
the
 
interaction.
 
6.
 
Data
 
may
 
be
 
obtained
 
from
 
backend
 
REST
 
services.
 
7.
 
The
 
model
 
is
 
updated.
 
8.
 
The
 
updated
 
information
 
is
 
displayed
 
through
 
the
 
view.
 
AngularJS
 
therefore
 
provides
 
a
 
clear
 
separation
 
between
 
the
 
functional
 
layers
 
of
 
an
 
application.
 
Conclusion
 
AngularJS
 
MVC
 
consists
 
of
 
Model,
 
View,
 
and
 
Controller
.
 
The
 
model
 
uses
 
$scope
 
to
 
store
 
application
 
data,
 
the
 
view
 
uses
 
HTML


In [ ]:
@tool
def skill_gap_tool(
    current_skills: str,
    target_role: str
) -> str:
    """Compare student skills with skills required for a target job."""

    docs = retriever.invoke(
        f"Required skills for {target_role}"
    )

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""
You are a Skill Gap Analysis Assistant.

Target Job Role:
{target_role}

Student's Current Skills:
{current_skills}

Required Job Information:
{context}

Compare the student's current skills with the
skills required for the target role.

Give the result in these sections:

1. Skills Already Have
2. Missing Skills
3. Partially Matching Skills
4. Suggested Learning Areas

Use only the provided job information.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
result = skill_gap_tool.invoke({
    "current_skills":
        "Python, HTML, CSS, SQL",
    "target_role":
        "Data Scientist"
})

print(result)

**1. Skills Already Have**  
- **Python** – already known; useful for backend logic or data handling, though not mentioned in the AngularJS‑focused requirements.  
- **HTML** – you can write the markup that AngularJS uses as view templates.  
- **CSS** – you can style the HTML templates that AngularJS renders.  
- **SQL** – you can work with data stores that a backend REST service might query, even though the job description does not call for it directly.  

**2. Missing Skills**  
- **AngularJS core library** (`angular.min.js`)  
- **AngularJS routing module** (`angular-route.min.js`) – required for SPA navigation.  
- **AngularJS cookies module** (`angular-cookies.min.js`) – for client‑side state management.  
- **AngularJS resource module** (`angular-resource.min.js`) – provides REST service integration.  
- **jQuery** – the supporting front‑end JavaScript library used together with AngularJS.  
- **MVC (Model‑View‑Controller) architecture** – understanding how the model (`$scope`),

In [ ]:
@tool
def learning_roadmap_tool(
    missing_skills: str
) -> str:
    """Create a learning roadmap for missing skills."""

    prompt = f"""
Create a simple learning roadmap for these
missing skills:

{missing_skills}

Organize it as:

1. Beginner
2. Intermediate
3. Advanced

Give a logical learning order.
Keep the explanation suitable for a college student.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
result = learning_roadmap_tool.invoke({
    "missing_skills":
        "Statistics, Pandas, Machine Learning, Data Visualization"
})

print(result)

## 📚 Simple Learning Road‑Map  
**Goal:** Build a solid foundation in data‑science fundamentals and progress to being able to design, implement, and communicate real‑world ML projects.  

| Level | What to Learn (in order) | Why it matters | Quick Resources |
|-------|--------------------------|----------------|-----------------|
| **1️⃣ Beginner** | 1. **Basic Statistics** – descriptive stats, probability basics, distributions, hypothesis testing. <br>2. **Python fundamentals** – data types, loops, functions, list‑comprehensions, virtual environments. <br>3. **Intro to Pandas** – Series, DataFrames, reading/writing CSV/Excel, basic indexing & slicing, `head()/describe()`. <br>4. **Simple Data‑Viz** – Matplotlib & Seaborn basics (line, bar, histogram, scatter). | You need to *understand data* before you can manipulate or model it. Pandas is the work‑horse for cleaning data; Matplotlib/Seaborn let you see patterns early. | • “Khan Academy – Statistics” (free videos) <br>• “Python for Ev

In [ ]:
from langchain.agents import create_agent

tools = [
    skill_gap_tool,
    learning_roadmap_tool
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a Skill Gap Analysis Agent.

Your purpose is to help students understand
what skills they need for their target career.

Use the Skill Gap Analysis Tool to compare
current skills with job requirements.

Use the Learning Roadmap Tool to create a
learning plan for missing skills.

Ask for the student's current skills and
target role when necessary.

Give clear and simple answers.
"""
)

print("Agent created successfully")

Agent created successfully


In [ ]:
chat_history = []

In [ ]:
def ask_agent(message):

    chat_history.append({
        "role": "user",
        "content": message
    })

    response = agent.invoke({
        "messages": chat_history
    })

    answer = response["messages"][-1].content

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [ ]:
def chatbot(message, history):

    chat_history.append({
        "role": "user",
        "content": message
    })

    response = agent.invoke({
        "messages": chat_history
    })

    answer = response["messages"][-1].content

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="🎯 Skill Gap Analysis Agent",
    description="Enter your skills and target career to identify skill gaps and create a learning roadmap."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee6b9a0543b2688cb7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
